# Playground — central orchestrator

Linear data flow:
1. **Load** raw data (both Excel sheets)
2. **Clean** (split sales / returns)
3. **Features** (weekly aggregation, return rate, calendar, lags, price, demand-class)
4. **Embedding** (Gemini Embedding 2 on canonical descriptions, parquet-cached)
5. **Clustering** (HDBSCAN on `[UMAP(emb) ⊕ demand profile ⊕ commercial profile]`)
6. Model selection on rolling-origin folds (Naive / SARIMAX / Prophet / LightGBM)
7. Final 12-week forecast + revenue translation
8. Plots
9. **Agent artifacts** (parquet) consumed by `agent/server.py` for n8n

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import pandas as pd
import matplotlib.pyplot as plt

from src.tools import (
    load_raw_data, split_sales_returns,
    aggregate_weekly_sku, median_price_per_sku,
    eligible_skus_by_revenue, return_rate_features,
    add_calendar_features, add_lag_rolling_features, add_price_features,
    demand_classification, commercial_profile,
    embed_sku_descriptions,
    cluster_skus, cluster_summary,
)
from src.modelling import default_registry, select_best_model, forecast_final_horizon, attach_revenue
from agent import build_agent_tables, get_forecast_for_prompt

## 1. Load — both Excel sheets

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "online_retail_II.xlsx"
df = load_raw_data(DATA_PATH)
print(f"raw rows: {len(df):,}  |  sheets: {df['SourceSheet'].unique().tolist()}")

## 2. Clean — sales (target) vs returns (feature only)

In [ ]:
sales, returns = split_sales_returns(df)
print(f"sales: {len(sales):,}  |  returns: {len(returns):,}  |  unique SKUs: {sales['StockCode'].nunique():,}")

## 3. Features — weekly + return rate + calendar + lags + price + demand class

Each builder takes the previous frame and adds columns. Static profiles (per SKU) are joined separately.

In [ ]:
weekly_sku = aggregate_weekly_sku(sales)
sku_price = median_price_per_sku(sales)

# dynamic features (per (SKU, Week))
feat = return_rate_features(weekly_sku, returns, windows=(4, 13))
feat = add_calendar_features(feat)
feat = add_lag_rolling_features(feat)
feat = add_price_features(feat, sales)

# static features (per SKU)
demand_cls = demand_classification(weekly_sku)
commercial = commercial_profile(sales)

TOP_N = 30
skus = eligible_skus_by_revenue(weekly_sku, top_n=TOP_N, min_active_weeks=70)  # enough for SARIMAX seasonal
print(f"weekly rows: {len(weekly_sku):,}  |  feat columns: {feat.shape[1]}  |  SKUs to evaluate: {len(skus)}")
demand_cls['demand_class'].value_counts()

## 4. Embedding — Gemini Embedding 2 on canonical SKU descriptions

Requires `GEMINI_API_KEY` in `.env`. First run hits the API; subsequent runs read parquet.

In [ ]:
EMB_PATH = ROOT / "data" / "processed" / "sku_desc_emb.parquet"
emb = embed_sku_descriptions(sales, cache_path=EMB_PATH, dim=768, task_type="CLUSTERING")
print(f"embedded SKUs: {len(emb):,}  |  dim: {len(emb['embedding'].iloc[0])}")
emb.head(3)

## 5. Clustering — HDBSCAN on [UMAP(emb) ⊕ demand profile ⊕ commercial profile]

Output `cluster_id` enters DeepAR / NS-Transformer as `static_cat`. Outliers are labeled `-1`.

In [ ]:
clusters = cluster_skus(emb, demand_cls, commercial, umap_dim=32, min_cluster_size=20)
print(cluster_summary(clusters).head(10))

## 6. Model selection — rolling-origin (3 folds × 12-week test)

Fold 1 = validation (pick best model per SKU). Folds 2..N = held-out test. With ~106 weeks, fold 1 train ≥ 70 weeks → enough for SARIMAX seasonal s=52.

Add models by extending the registry — each is `(train, horizon) -> ndarray`.

In [ ]:
registry = default_registry()
print("Models:", list(registry))

results = select_best_model(
    weekly_sku=weekly_sku, skus=skus, model_registry=registry,
    n_folds=3, test_size=12, block_size=4, min_train=70,
)
choices_df = results["choices"]
test_block = results["test_block"]
if not test_block.empty:
    print("Test block-MAPE per fold:", test_block.groupby('Fold')['Block_APE'].mean().round(2).to_dict())
choices_df.head()

## 7. Final 12-week forecast + revenue translation

In [ ]:
forecast_df = forecast_final_horizon(weekly_sku, choices_df, registry, horizon=12)
fc_with_rev = attach_revenue(forecast_df, sku_price)
fc_with_rev.head()

## 8. Plots

In [ ]:
from src.tools import plots
if not test_block.empty:
    plots.plot_block_ape_boxplot(test_block); plt.show()
plots.plot_chosen_model_counts(choices_df); plt.show()

## 9. Agent artifacts — parquet for the n8n FastAPI server

`agent/server.py` reads these on startup and exposes `/forecast` for n8n.
Boot the server with `uvicorn agent.server:app --port 8000` after this cell.

In [ ]:
agent_horizon, agent_summary = build_agent_tables(forecast_df, sku_price, choices_df)

OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
agent_horizon.to_parquet(OUT / "agent_horizon.parquet", index=False)
agent_summary.to_parquet(OUT / "agent_summary.parquet", index=False)
print(f"wrote {OUT}/agent_horizon.parquet, agent_summary.parquet")

# quick smoke test of the prompt handler
for prompt in [
    "What is the latest 12-week forecast for product 22197?",
    "Forecast SKU 85123A for the next 4 weeks",
]:
    print(get_forecast_for_prompt(prompt, agent_summary, agent_horizon))
    print("=" * 60)